In [1]:
import requests
import pandas as pd

In [2]:
API_KEY = "OPENROUTER_API_KEY"

URL = "https://openrouter.ai/api/v1/chat/completions"

MODEL = "google/gemini-2.5-flash-lite"

In [3]:
def call_gemini(prompt):

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    response = requests.post(
        URL,
        headers=headers,
        json=data
    )

    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"].strip()

In [5]:
messages = [

    {
        "message":"My package has not arrived yet.",
        "expected_category":"delivery"
    },

    {
        "message":"I paid for my subscription but it is still inactive.",
        "expected_category":"payment"
    },

    {
        "message":"I want my money back.",
        "expected_category":"refund"
    },

    {
        "message":"The application crashes after login.",
        "expected_category":"technical_support"
    },

    {
        "message":"I forgot my password.",
        "expected_category":"account"
    },

    {
        "message":"How can I change my email address?",
        "expected_category":"account"
    },

    {
        "message":"The courier delivered the wrong item.",
        "expected_category":"delivery"
    },

    {
        "message":"My card was charged twice.",
        "expected_category":"payment"
    },

    {
        "message":"I received a damaged product and want a refund.",
        "expected_category":"refund"
    },

    {
        "message":"The website shows Error 500.",
        "expected_category":"technical_support"
    },

    {
        "message":"Can I delete my account?",
        "expected_category":"account"
    },

    {
        "message":"What are your business hours?",
        "expected_category":"other"
    }

]

In [6]:
def zero_shot_prompt(message):

    return f"""
You are a customer support classifier.

Classify the customer message into ONLY ONE category.

Possible categories:
- delivery
- payment
- refund
- technical_support
- account
- other

Return ONLY the category name.

Customer message:
{message}
"""

In [7]:
def few_shot_prompt(message):

    return f"""
You are a customer support classifier.

Classify the customer message into ONLY ONE category.

Possible categories:
- delivery
- payment
- refund
- technical_support
- account
- other

Examples:

Message: My package has not arrived yet.
Category: delivery

Message: My payment was charged twice.
Category: payment

Message: I want my money back.
Category: refund

Message: The application crashes every time I open it.
Category: technical_support

Message: I forgot my password.
Category: account

Message: Thank you for your help!
Category: other

Now classify this message.

Message:
{message}

Return ONLY the category name.
"""

In [8]:
zero_shot_results = []

for item in messages[:7]:

    result = call_gemini(
        zero_shot_prompt(item["message"])
    )

    zero_shot_results.append(result)

zero_shot_results

['delivery',
 'payment',
 'refund',
 'technical_support',
 'account',
 'account',
 'delivery']

In [9]:
few_shot_results = []

for item in messages[:7]:

    result = call_gemini(
        few_shot_prompt(item["message"])
    )

    few_shot_results.append(result)

few_shot_results

['delivery',
 'payment',
 'refund',
 'technical_support',
 'account',
 'account',
 'delivery']

In [10]:
results = pd.DataFrame(messages[:7])

results["zero_shot_category"] = zero_shot_results
results["few_shot_category"] = few_shot_results

results["zero_shot_correct"] = (
    results["expected_category"] ==
    results["zero_shot_category"]
)

results["few_shot_correct"] = (
    results["expected_category"] ==
    results["few_shot_category"]
)

results

,message,expected_category,zero_shot_category,few_shot_category,zero_shot_correct,few_shot_correct
0,My package has not arrived yet.,delivery,delivery,delivery,True,True
1,I paid for my subscription but it is still ina...,payment,payment,payment,True,True
2,I want my money back.,refund,refund,refund,True,True
3,The application crashes after login.,technical_support,technical_support,technical_support,True,True
4,I forgot my password.,account,account,account,True,True
5,How can I change my email address?,account,account,account,True,True
6,The courier delivered the wrong item.,delivery,delivery,delivery,True,True


In [11]:
zero_accuracy = results["zero_shot_correct"].mean()

few_accuracy = results["few_shot_correct"].mean()

print(f"Zero-shot accuracy: {zero_accuracy:.2%}")
print(f"Few-shot accuracy: {few_accuracy:.2%}")

Zero-shot accuracy: 100.00%
Few-shot accuracy: 100.00%


In [12]:
def support_prompt(message):

    return f"""
You are a customer support assistant.

Rules:

- Be polite.
- Answer briefly.
- Do not invent facts.
- If you do not know something, tell the customer that support will investigate.
- Maximum 3 sentences.

Customer message:

{message}
"""

In [13]:
examples = [

    messages[0]["message"],
    messages[3]["message"],
    messages[8]["message"]

]

for text in examples:

    print("=" * 70)
    print("Customer:")
    print(text)
    print()

    answer = call_gemini(
        support_prompt(text)
    )

    print("Assistant:")
    print(answer)
    print()

Customer:
My package has not arrived yet.

Assistant:
I am sorry to hear your package has not arrived. Could you please provide your order number so I can check its status? Support will investigate if there are any delays.

Customer:
The application crashes after login.

Assistant:
I'm sorry to hear you're experiencing crashes after logging in. Our support team will investigate this issue promptly. We'll reach out with an update as soon as possible.

Customer:
I received a damaged product and want a refund.

Assistant:
I'm sorry to hear that your product arrived damaged. Please provide me with your order number so I can assist you further. Once I have that, I can initiate the refund process for you.



1. Few-shot приклади допомогли моделі точніше визначати категорії звернень, оскільки вона бачила правильні приклади класифікації перед виконанням завдання.
2. Найчастіше модель плутала схожі категорії, наприклад payment і refund або account і technical_support.
3. Few-shot prompt виявився стабільнішим, ніж zero-shot, тому що завдяки прикладам модель краще розуміла, як саме потрібно класифікувати повідомлення.
4. Prompt engineering достатньо використовувати для невеликих проєктів, прототипів або задач із невеликою кількістю категорій.
5. Якщо потрібно працювати з великою кількістю внутрішніх документів, специфічними знаннями компанії або досягти максимальної точності, краще використовувати RAG або fine-tuning моделі.